In [55]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/OCR_images/

total 1676
-rw------- 1 root root  12861 Jul 28 00:11 paper-cash-sell-receipt-vector.jpg
-rw------- 1 root root 181352 Jul 28 00:36 wallmart-reciept-hands.jpg
-rw------- 1 root root 185819 Jul 28 00:17 walmart_2.png
-rw------- 1 root root  47283 Jul 27 00:54 walmart-receipt.png
-rw------- 1 root root 458428 Jul 28 01:01 wendys-reciept-2.jpg
-rw------- 1 root root  36455 Jul 28 00:45 wendys-reciept-3.jpg
-rw------- 1 root root 791657 Jul 28 00:44 wendys-reciept.jpg


In [57]:
!wget https://ollama.com/install.sh

--2026-07-28 01:02:57--  https://ollama.com/install.sh
Resolving ollama.com (ollama.com)... 34.36.133.15
Connecting to ollama.com (ollama.com)|34.36.133.15|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: https://github.com/ollama/ollama/releases/latest/download/install.sh [following]
--2026-07-28 01:02:57--  https://github.com/ollama/ollama/releases/latest/download/install.sh
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/ollama/ollama/releases/download/v0.32.5/install.sh [following]
--2026-07-28 01:02:57--  https://github.com/ollama/ollama/releases/download/v0.32.5/install.sh
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/658928958/e9881ec6-5898-4bf6-9930-6e

In [58]:
!nvidia-smi

Tue Jul 28 01:02:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P0             29W /   70W |    6651MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [59]:
import os

# Set the system path environment variable so Ollama can find the T4 libraries
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

!chmod +x install.sh
!sudo apt-get install zstd
!./install.sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [60]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print("Ollama is up and running")

Ollama is up and running


In [61]:
!ollama pull qwen2.5vl

In [62]:
!pip install ollama

In [63]:
import ollama

image_path = '/content/drive/MyDrive/OCR_images/wendys-reciept-2.jpg'

# The Agentic System Prompt guiding the VLM's behavior
system_instruction = (
    "You are an expert autonomous AI accountant. Analyze the provided invoice image. "
    "Perform the following tasks:\n"
    "1. Extract the Vendor Name, Invoice Date, and Total Amount.\n"
    "2. Check the line items. Manually multiply the quantities by unit prices "
    "and verify if the mathematical total matches the listed total.\n"
    "3. Audit the document: Flag any suspicious discrepancies, missing tax details, "
    "or mathematical errors.\n"
    "4. Output your final decision: 'APPROVED' or 'FLAGGED FOR REVIEW' with a clear reason."
)

# Send image data locally within the Colab VM environment
response = ollama.chat(
    model='qwen2.5vl',
    messages=[
        {
            'role': 'user',
            'content': system_instruction,
            'images': [image_path]
        }
    ]
)

print(response['message']['content'])

### Analysis of the Invoice Image

#### 1. Extract Vendor Name, Invoice Date, and Total Amount:
- **Vendor Name:** Not explicitly stated on the receipt; it appears to be "KFC" based on the logo.
- **Invoice Date:** Not visible in the image.
- **Total Amount:** $41.07 (Dine In Total)

#### 2. Check Line Items:
The line items and their prices are as follows:

- Single Cheese: $3.99
- Medium Fries: $3.29
- MD Drink: $3.59
- Large Fries (@$3.29): $13.16 (4 x 3.29)
- Spicy Nuggets (10 pc): $3.79
- Sweet & Sour (2): $2.19

Let's verify the total:
- Single Cheese: $3.99
- Medium Fries: $3.29
- MD Drink: $3.59
- Large Fries (@$3.29): $13.16 (4 x 3.29)
- Spicy Nuggets (10 pc): $3.79
- Sweet & Sour (2): $2.19

Summing these up:
\[ 3.99 + 3.29 + 3.59 + 13.16 + 3.79 + 2.19 = 30.43 \]

The total listed on the receipt is $37.72, which does not match our calculated sum of $30.43.

#### 3. Audit Document:
- **Discrepancy:** The mathematical total from line items ($30.43) does not match the listed tota